### Crop pictures

This will create a new folder called Poles_cropped, at the same location as Poles. 
It will contain only the rgb folder, where all files are cropped, so that the top 40% is removed from all pictures
(And txt files are changed accordingly)

In [ ]:
import os
import glob
import cv2
import numpy as np
import shutil
from tqdm import tqdm

# Dataset path
dataset_path = "/home/omtalmo/Olaf_TTK4265/Poles/rgb"
cropped_dataset_path = "/home/omtalmo/Olaf_TTK4265/Poles_cropped/rgb"

def create_directory_structure(cropped_dataset_path):
    """
    Create the directory structure for the cropped dataset
    
    Args:
        cropped_dataset_path: Path to the new cropped dataset directory
    """
    # Create main directories
    os.makedirs(cropped_dataset_path, exist_ok=True)
    os.makedirs(os.path.join(cropped_dataset_path, "images"), exist_ok=True)
    os.makedirs(os.path.join(cropped_dataset_path, "labels"), exist_ok=True)
    
    # Create subdirectories
    for split in ["train", "valid", "test"]:
        os.makedirs(os.path.join(cropped_dataset_path, "images", split), exist_ok=True)
    
    for split in ["train", "valid"]:  # No labels for test set
        os.makedirs(os.path.join(cropped_dataset_path, "labels", split), exist_ok=True)
    
    # Copy data.yaml file with updated paths
    data_yaml_path = os.path.join(dataset_path, "data.yaml")
    if os.path.exists(data_yaml_path):
        with open(data_yaml_path, 'r') as f:
            data_yaml_content = f.read()
        
        # Update paths in the data.yaml
        data_yaml_content = data_yaml_content.replace(
            "home/omtalmo/Olaf_TTK4265//Poles/rgb", 
            "home/omtalmo/Olaf_TTK4265//Poles_cropped/rgb"
        )
        
        with open(os.path.join(cropped_dataset_path, "data.yaml"), 'w') as f:
            f.write(data_yaml_content)
    
    print("Directory structure created")

def crop_images(original_path, cropped_path, crop_percent=0.4):
    """
    Crop images and save to new directory
    
    Args:
        original_path: Path to the original dataset directory
        cropped_path: Path to the new cropped dataset directory
        crop_percent: Percentage of the top portion to remove (0.4 = 40%)
    """
    # Define directories containing images
    image_splits = ["train", "valid", "test"]
    
    # Process each directory
    for split in image_splits:
        img_dir = os.path.join(original_path, "images", split)
        output_dir = os.path.join(cropped_path, "images", split)
        
        if not os.path.exists(img_dir):
            print(f"Directory not found: {img_dir}")
            continue
            
        # Get all PNG images in the directory
        image_paths = glob.glob(os.path.join(img_dir, "*.PNG"))
        
        print(f"Processing {len(image_paths)} images in {split} set...")
        
        # Process each image
        for img_path in tqdm(image_paths):
            # Read the image
            img = cv2.imread(img_path)
            
            if img is None:
                print(f"Could not read image: {img_path}")
                continue
                
            # Get image dimensions
            height, width = img.shape[:2]
            
            # Calculate crop dimensions (remove top 40%)
            start_y = int(height * crop_percent)
            
            # Crop the image
            cropped_img = img[start_y:height, 0:width]
            
            # Save cropped image to new location
            img_filename = os.path.basename(img_path)
            output_path = os.path.join(output_dir, img_filename)
            cv2.imwrite(output_path, cropped_img)
    
    print("Image cropping complete!")

def process_labels(original_path, cropped_path, crop_percent=0.4):
    """
    Process label files for the cropped images
    
    Args:
        original_path: Path to the original dataset directory
        cropped_path: Path to the new cropped dataset directory
        crop_percent: Percentage of the top portion removed (0.4 = 40%)
    """
    label_splits = ["train", "valid"]  # No labels for test set
    
    for split in label_splits:
        label_dir = os.path.join(original_path, "labels", split)
        output_dir = os.path.join(cropped_path, "labels", split)
        
        if not os.path.exists(label_dir):
            print(f"Directory not found: {label_dir}")
            continue
            
        # Get all TXT files
        label_paths = glob.glob(os.path.join(label_dir, "*.txt"))
        
        print(f"Processing {len(label_paths)} labels in {split} set...")
        
        # Process each label file
        for label_path in tqdm(label_paths):
            try:
                # Read label file
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                
                # Process and adjust each bounding box
                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        class_id = parts[0]
                        x_center = float(parts[1])
                        y_center = float(parts[2])
                        width = float(parts[3])
                        height = float(parts[4])
                        
                        # Adjust y_center based on crop
                        # y coordinate is normalized to [0,1] so we need to:
                        # 1. Subtract crop_percent from numerator
                        # 2. Divide by the new height (1-crop_percent)
                        new_y_center = (y_center - crop_percent) / (1 - crop_percent)
                        
                        # Adjust height based on crop
                        new_height = height / (1 - crop_percent)
                        
                        # Only keep boxes that are still visible after cropping
                        if new_y_center >= 0 and new_y_center <= 1 and (new_y_center + new_height/2) <= 1 and (new_y_center - new_height/2) >= 0:
                            new_line = f"{class_id} {x_center} {new_y_center} {width} {new_height}\n"
                            new_lines.append(new_line)
                
                # Write processed labels to new location
                label_filename = os.path.basename(label_path)
                output_path = os.path.join(output_dir, label_filename)
                with open(output_path, 'w') as f:
                    f.writelines(new_lines)
                    
            except Exception as e:
                print(f"Error processing {label_path}: {str(e)}")
    
    print("Label processing complete!")

def main(crop_percent=0.4):
    """
    Main function to create cropped dataset
    
    Args:
        crop_percent: Percentage of top to crop (0.4 = 40%)
    """
    print(f"Creating cropped dataset with {crop_percent*100}% top cropping")
    
    # Step 1: Create directory structure
    create_directory_structure(cropped_dataset_path)
    
    # Step 2: Crop and save images
    crop_images(dataset_path, cropped_dataset_path, crop_percent)
    
    # Step 3: Process label files
    process_labels(dataset_path, cropped_dataset_path, crop_percent)
    
    print(f"Cropped dataset created at: {cropped_dataset_path}")

# Execute the preprocessing
main(crop_percent=0.4)

Creating cropped dataset with 40.0% top cropping
Directory structure created
Processing 322 images in train set...


100%|██████████| 322/322 [00:18<00:00, 17.22it/s]


Processing 92 images in valid set...


100%|██████████| 92/92 [00:05<00:00, 17.54it/s]


Processing 46 images in test set...


...

Image cropping complete!
Processing 322 labels in train set...


100%|██████████| 322/322 [00:00<00:00, 475.76it/s]


Processing 92 labels in valid set...


100%|██████████| 92/92 [00:00<00:00, 396.75it/s]

Label processing complete!
Cropped dataset created at: /home/omtalmo/Olaf_TTK4265/Poles_cropped/rgb
